# Signal & Background Distributions (Before Cosmic Tagger)

Stacked SIGNAL-vs-BACKGROUND distributions for the charge-light matching
pipeline, drawn on the same true-cluster population
`Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb` evaluates: the same
input tree and the same true-side cuts.

**The stack, so far**

| component | definition |
|---|---|
| `numu_CC_in_volume` | true neutrino clusters whose interaction is numu CC and whose vertex is inside the wire-readout sensitive volume -- the signal |
| `nue_CC_in_volume` | the same, for nue CC -- a background: a CC interaction of the wrong flavour, depositing the same kind of energy as the signal |
| `NC_in_volume` | the same, for NC -- a background with no charged lepton, so it deposits less of the incident energy |
| `neutrino_out_of_volume` | any channel, vertex OUTSIDE the volume -- energy that leaked in from an interaction the analysis does not accept |

Those four are every true neutrino cluster whose vertex is known: the three
channels partition the interaction types and in/out partitions the volume. A
neutrino with no mc.json vertex has `vertex_in_volume=None` and is claimed by
nothing -- unknown, rather than assigned to a side. Cosmic clusters, most of the
sample, are still outside the stack.

Further components are added by appending to `SIGNAL_BACKGROUND_COMPONENTS` in
`AnalysisDistributions/draw_signal_background.py`; the drawing code, the legend
and the text table are all driven off that list, so nothing here changes when
one arrives. A one-component stack draws as a single filled histogram, on the
axes and binning the backgrounds will be added to.

**The plot**

| axis | quantity |
|---|---|
| x | TRUE DEPOSITED energy of the true cluster (MeV, 200 MeV bins, fixed 0-5000 axis) |
| y | number of clusters |

The x variable is the sum of sed-smear's per-point `e` over the cluster --
energy that reached the argon and survived the selection, *not* the incident
neutrino energy. An interaction that put half its energy into an escaping
neutron appears here at what the detector saw. Bins are a uniform 200 MeV from
zero (`draw_signal_background.ENERGY_BIN_WIDTH_MEV`) -- wider than the 100 MeV
`Reco_Distributions.ipynb` uses, because a stack is read by comparing band
thicknesses within a bin and narrower bins hold one or two clusters here. The
module asserts on import that the width is a whole multiple of that 100 MeV, so
each bin is exactly two of theirs and the edges never straddle.

y counts CLUSTERS. Neutrino clusters are reassigned to `99990+nu_idx`, one per
interaction, so for the components here a cluster *is* an interaction -- which
stops being true the moment a cosmic component is added, hence the axis label.

**THE RECO OVERLAY**

**No reco clusters are drawn.** These are stacked histograms of TRUE clusters,
and a reco cluster is a different kind of object: it carries no truth label, and
one true cluster split into two reco clusters counts once here and twice there.
Putting the two populations on one axes -- stacked or overlaid -- invites a
comparison neither supports. `draw_stacked_true_energy` still accepts
`reco_records` and will draw them as a step curve if asked; this notebook does
not ask.

Reco clusters carry charge, not energy. Nothing here plots them, but the
conversion is kept in `draw_signal_background.py` for a reco-side figure of its
own, and the reco populations are still counted in `summary.txt`:

> `E_deposited = 23.6 eV * Q / 0.7`

-- the LAr ionisation work function over the recombination survival fraction,
putting back the charge that recombined. It assumes the charge unit is one
electron, one recombination factor for every track, and no losses to lifetime,
dead channels or clustering. Those last two pull opposite ways and the net is
that this reads ~12-26% HIGH against the fitted calibration in
`EnergyReconstruction/fit_energy_calibration.py`; see the constants block in
`draw_signal_background.py` for the full accounting.

**Two plots are drawn, one per reco selection.** The stack is identical in both
-- the true side is never touched -- so what changes between them is only which
reco clusters the black curve counts:

| selection | reco clusters counted |
|---|---|
| `NoCuts` | every reco cluster in the event: no beam-window cut, no fiducial cut, no minimum point count |
| `AfterBeamWindowCut` | only those whose bridged optical flash time falls inside `[BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US]`. A cluster with **no** flash is cut -- it has no time, so it cannot be shown to be in the window. Same cut, and the same two constants, as `Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb` |

The label goes in the title, the legend and both output filenames, so the two
versions sit side by side in `job_summary/` instead of overwriting each other.
Comparing them is the point: the beam-window cut is what removes the cosmic
pile-up, and these two plots show how much of it goes and how much true signal
the surviving reco population has to be found among.

The TRUE side is unchanged and still carries the evaluation's cuts: energy
cutoff, wire-readout fiducial cut, and the dead-area cut baked into the input
tree.

**A third plot, `NumuCCQuality`,** splits the signal band by how well each
cluster was reconstructed:

| band | definition |
|---|---|
| high completeness | its 1-to-1 matched reco cluster has **both** completeness and purity above `NUMU_QUALITY_THRESHOLD` (80%) |
| contaminated | every other numu CC in-volume cluster -- low completeness, low purity, both, **or never matched at all** |

The second is defined as the *complement* of the first, so the two always
partition the signal exactly and no cluster can fall between them. Note this
puts unmatched clusters in `contaminated`: they have no pair, so their
completeness and purity are `None` rather than zero, and the band therefore
mixes "reconstructed badly" with "not reconstructed".

That split is the one thing here that needs the **1-to-1 pairing**, so the
notebook runs `EvaluateCompleteness`, `EvaluatePurity` and `MatchTrueToReco1to1`
-- against the **beam-window** reco clusters, since that is the population these
plots show, with the same radii as
`Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb`. The other two plots do
not depend on it.

**Output** -- JOB LEVEL ONLY

`AnalysisDistributions/multi_file_plots_charge_light_matching/Signal_Background_Distributions/combined_apa_<date>_<time>/job_summary/`,
holding one stack PNG and one per-bin table per reco selection, plus the run
summary:

```
signal_background_true_energy_stack_NoCuts_job_Combined.png
signal_background_true_energy_stack_AfterBeamWindowCut_job_Combined.png
signal_background_info_NoCuts.txt
signal_background_info_AfterBeamWindowCut.txt
summary.txt
```

There are no per-file or per-event directories. A stack is a statement about a
composition, and an event holds one or two signal clusters -- a per-event stack
would be a bar of height one, and a per-file one is still too thin to read a
composition off. Everything is pooled into the job-level figure instead.


In [ ]:
# Run scope -- same knobs as Reco_Distributions.ipynb and
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb. Charge-light matching
# is a combined-APA evaluation (img-global / sed-sce are already global across
# APAs) -- no per-APA/face looping.
files     = "all"   # "all", or 1/2/3/... to limit the number of file subdirectories processed
events    = "all"   # "all", or 1/2/3/... to limit the number of events processed per file

# ========================================================================
# SELECTIVE FILE/EVENT FILTERING (Optional)
# ========================================================================
# Set to None to process all files/events, or specify to run only specific ones
# Example: target_file = "file0", target_event = 3  (to process only file0, event 3)
target_file  = None   # Set to "file0", "file1", etc. to process specific file only
target_event = None   # Set to a SINGLE event number (0, 1, ..., 9); use target_event_range below for a span

# Range of event numbers to run, inclusive on both ends: (1, 5) runs events
# 1,2,3,4,5. None runs every event. Applied on top of target_event, so leave
# target_event = None when using a range.
target_event_range = None   # e.g. (1, 5) for events 1..5

# Range of file INDICES to run, inclusive on both ends: (6, 9) runs file6, file7,
# file8, file9. None runs every file. Matched on the number at the end of the
# directory name, NOT on position in the list -- the directories sort
# lexicographically (file0, file1, file10, file11, file2, ...), so a positional
# slice would pick the wrong files. The `files = N` knob above still takes the
# first N in lexicographic order.
# NOTE: target_file (above) is applied too, so set it to None when using a range,
# otherwise only the one file that satisfies both runs.
target_file_range = None   # e.g. (6, 9) for file6..file9

# Fail fast rather than silently processing nothing: `evt != target_event` can
# never be False for a tuple, so target_event = (1, 5) would skip every event.
if isinstance(target_event, (tuple, list)):
    raise ValueError(
        f"target_event={target_event} is a range, but target_event takes a single event number. "
        f"Use target_event_range={tuple(target_event)} and target_event = None instead.")


# ========================================================================
# NO LEVEL SWITCHES -- this notebook draws the JOB-LEVEL stack only.
# ========================================================================
# Unlike Reco_Distributions.ipynb there is no b_draw_event/file_level_plots
# knob. A stack answers "what is this sample made of", which needs the pooled
# statistics to mean anything: per event there are one or two signal clusters,
# so an event-level stack is a single bar of height one. The file/event loops
# below still run -- they are how the clusters are collected -- they just do not
# draw or write anything of their own.


In [ ]:
%load_ext autoreload
%autoreload 2

# python libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import os
import time
from datetime import datetime

np.set_printoptions(linewidth=1000)

# This notebook lives in AnalysisDistributions/, one level below the
# repository root where the pipeline modules and the input trees are. Resolve
# both explicitly so the notebook runs whether Jupyter was started in this
# directory (the usual case) or at the repository root.
NB_DIR = Path.cwd()
if NB_DIR.name != "AnalysisDistributions":
    NB_DIR = NB_DIR / "AnalysisDistributions"
NB_DIR = NB_DIR.resolve()
REPO_ROOT = NB_DIR.parent

for path in (str(REPO_ROOT), str(NB_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

# Record job start time (used to report total job runtime at the end)
job_start_time = time.time()
print(f"Job started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Notebook directory: {NB_DIR}")
print(f"Repository root:    {REPO_ROOT}")


In [ ]:
# Pipeline modules (repository root) -- imported and used UNCHANGED. Only the
# TRUE side of the chain is needed here (see the header): the reco clusters, the
# beam-window cut, completeness/purity and the 1-to-1 pairing are all absent
# because no quantity on these plots depends on them.
from readfiles import (ensure_data_extracted, stage_nuecc_chunks,
                       read_charge_light_files_for_event, flatten_mc_tree)
from selections import (
    GroupClustersByID, build_true_points_charge_light, apply_deadarea_cut_true_charge_light,
    reassign_cluster_ID_true_charge_light,
    apply_energy_cutoff, apply_true_pointwise_energy_cutoff,
    apply_energy_cutoff, apply_min_true_points_cutoff,
    apply_wire_readout_sensitive_yz_plane_cut_true,
    apply_cosmic_tagger_cut,
)
from cluster_category import cluster_category
from completeness_purity_estimate import EvaluateCompleteness, EvaluatePurity
from clusterpairmatching import MatchTrueToReco1to1
from metadata import (
    build_true_cluster_type_records, build_neutrino_vertex_records,
    build_cluster_flash_metadata, build_img_cluster_flash_metadata,
    add_metadata_true_reco_pair_cluster,
)
from DrawRecoTrueClusterCount import DrawRecoClusterSelectionFlow
from DrawRecoTrueFlashes import BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US

# Per-cluster record builder shared with Reco_Distributions.ipynb, so a cluster's
# total_energy means exactly the same thing in both notebooks.
from draw_variables import build_true_cluster_variable_records, build_reco_cluster_variable_records

# The new module for this notebook (AnalysisDistributions/draw_signal_background.py):
# the stack's component list, the channel join, the stacked drawer and the table.
from draw_signal_background import (
    SIGNAL_BACKGROUND_COMPONENTS, attach_interaction_channel, order_components_for_stack,
    reco_cluster_energy_mev, attach_pair_metrics, NUMU_QUALITY_THRESHOLD,
    BIN_WIDTHS_MEV,
    SIGNAL_BACKGROUND_COMPONENTS_NUMU_QUALITY,
    RECO_WORK_FUNCTION_EV, RECO_RECOMBINATION_FACTOR, DEFAULT_RECO_CUTS_LABEL,
)

# Selection performance: reco-space categorisation and its three plots.
from draw_selection_performance import (
    categorize_reco_clusters, draw_reco_selection_stack, draw_completeness_vs_purity,
    draw_completeness_vs_purity_colz,
    build_selection_efficiency, draw_selection_efficiency,
    write_selection_performance_info, write_selection_performance_root,
    write_efficiency_summary, write_efficiency_by_threshold_summary, EFFICIENCY_CURVE_SETS,
    draw_neutrino_multiplicity, draw_efficiency_by_multiplicity, MULTIPLICITY_CLASSES,
    plot_directory, EFFICIENCY_THRESHOLDS, draw_efficiency_threshold_comparison,
    UNCERTAINTY_STYLES, UNCERTAINTY_DIRS, count_signal_interactions_per_event,
    draw_energy_reconstruction, ENERGY_RECO_QUALITY,
    MULTIPLICITY_FIGURES,
    EFFICIENCY_BINNINGS, rebin_efficiency_tail,
    CHANNELS, MIN_MATCH_PURITY, HIGH_SIGNAL_THRESHOLD,
)

# Per-cluster XZ/YZ/XY views. Imported unconditionally (cheap), used only when
# SAVE_CLUSTER_VIEWS is on.
from draw_saved_clusters import (
    ClusterViewSampler, save_event_cluster_views, write_cluster_view_index,
)


In [ ]:
# Configuration: Parent directory containing multiple file subdirectories (file0/, file1/, ...)
#
# Expected structure (per file subdirectory). The preprocessed tree has no zip --
# its data/ is already present, so ensure_data_extracted() below simply no-ops.
# PARENT_DIR/
#   file0/data/0/0-sed-smear_readout.json                (true clusters)
#   file0/data/0/0-mc.json                               (particle truth ancestry tree)
#   file0/data/1/, 2/, ... (one subdirectory per event)
#
# The reco/optical files in the same tree (0-img-global.json,
# 0-clustering-global.json, 0-op.json) are read by read_charge_light_files_for_event
# but not used here -- this notebook is truth only.

# ========================================================================
# INPUT SAMPLE  (kept identical to Draw_Signal_Background_True.ipynb)
# ========================================================================
#   "tagger_100files" -> the MCP2025C Fall, Tagger-included 100-file production.
#       Its chunk zips extract to data/<event>/ at their own root, so each chunk
#       is unzipped into its OWN subdirectory (chunk0/, chunk1/, ...) to give the
#       <file>/data/<event>/ layout this loop expects -- a chunk then plays the
#       role a fileN directory plays elsewhere. ensure_data_extracted() does NOT
#       do this for you: it only matches mabc*.zip. THIS TREE IS RAW -- the
#       dead-area cut has NOT been applied to it, so Apply_deadarea_cut below
#       stays False by explicit choice and true clusters keep their points in
#       dead regions.
#
#   "nuecc" -> the img-clus-match-tag-pr-nuecc sample: ONE event per zip
#       (bee_r<run>_s<subrun>_e<event>.zip, contents at data/0/0-*.json; the "0"
#       is NOT the event number, the real (run, subrun, event) is in the zip
#       name). stage_nuecc_chunks() rewrites the first NUECC_N_FILES zips, in
#       groups of NUECC_CHUNK_SIZE, into chunk<NN>/data/<k>/<k>-*.json under
#       NUECC_STAGING_ROOT -- the same layout as tagger_100files, so the whole
#       loop below is unchanged. Each chunk<NN>/ gets an event_map.txt mapping
#       the renumbered k back to (run, subrun, event, zip). Same JSON schema and
#       same dead-area caveat as above.
SAMPLE = "nuecc"

# nuecc staging knobs -- only read when SAMPLE == "nuecc".
#
# The 8866 zips are split into bee/chunk_00 .. chunk_88 (89 dirs; chunk_88 has
# 66), each into subchunk_00.. of 10 -- see bee/chunk_manifest.txt. A job runs
# every chunk in NUECC_SOURCE_CHUNKS; each chunk's subchunks are staged into
# staging/<chunk>__<subchunk>/ (globally unique, so several chunks share one
# staging/ without colliding). NUECC_N_FILES is per source chunk.
# SAMPLE_NAME picks the nuecc-style production AND the per-sample output
# subdirectory, so nue-CC and numu runs land beside each other:
#   "NuECC_Sample"  -> img-clus-match-tag-pr-nuecc-1000file-2026-08-29   (nue CC)
#   "NuMuCC_Sample" -> img-clus-match-tag-pr-mc-1000file-sync-2026-08-30  (numu beam)
SAMPLE_NAME = "NuMuCC_Sample"
_SAMPLE_PRODUCTION = {
    "NuECC_Sample":  "img-clus-match-tag-pr-nuecc-1000file-2026-08-29",
    "NuMuCC_Sample": "img-clus-match-tag-pr-mc-1000file-sync-2026-08-30",
}
NUECC_BEE_ROOT      = (Path("/Volumes/My Passport/Research_Life/Experiment/SBND/"
                            "Wirecell_Reconstruction/Samples")
                       / _SAMPLE_PRODUCTION[SAMPLE_NAME] / "bee")
NUECC_SOURCE_CHUNKS = [f"chunk_{i:02d}" for i in range(15)]   # bee/chunk_NN dir(s) this job runs
NUECC_STAGING_ROOT  = NUECC_BEE_ROOT.parent / "staging"
NUECC_N_FILES       = 100  # per source chunk (10 = subchunk_00 only; 100 = whole chunk)
NUECC_CHUNK_SIZE    = 10    # events per staged subchunk

if SAMPLE == "nuecc":
    PARENT_DIR = NUECC_STAGING_ROOT
else:
    PARENT_DIR = REPO_ROOT / "Haiwang_files_charge_light_matching_Tagger_Included_MCP2025C_FallProd_100files"

# Number of files/events to process (convert the 'files'/'events' knobs above)
num_files_to_process  = None if files  == "all" else files
num_events_to_process = None if events == "all" else events

# Output directory: inside AnalysisDistributions, alongside
# RecoTrue_Distributions_AfterTimeWindowCut, so each notebook in this directory
# owns one subdirectory of the same plot tree.
PLOTBASEDIR = NB_DIR / "multi_file_plots_charge_light_matching" / "Signal_Background_Distributions_BeforeCosmicTagger" / SAMPLE_NAME
PLOTBASEDIR.mkdir(parents=True, exist_ok=True)

print("Configuration:")
print(f"Parent directory: {PARENT_DIR}")
print(f"Plot base directory: {PLOTBASEDIR}")
print(f"Files to process: {files}")
print(f"Events to process: {events}")

if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    print(f"\nSELECTIVE FILTERING ENABLED:")
    print(f"  Target file: {target_file if target_file else 'all'}")
    print(f"  Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        print(f"  Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        print(f"  Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")

# ========================================================================
# SELECTION PARAMETERS -- the TRUE-side subset of
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb's, deliberately
# identical: these distributions describe the true population that notebook
# evaluates, so any change here breaks that correspondence.
#
# The reco-side parameters (matching radii, min_reco_points_cutoff, the
# beam-window cut) are not here because no reco cluster is read.
# ========================================================================

# min_true_points_cutoff is DISABLED: this format's point clouds are much
# sparser than the old imaging-based reconstruction -- real neutrino clusters
# have been seen with as few as 13 points -- so the old threshold (200) would
# delete real signal clusters outright.
#
# min_cluster_energy is NOT applied (Apply_energy_cutoff = False below): only
# the per-POINT floor (min_true_point_energy) still cuts the true side, so a
# cluster's total can be any positive energy -- draw_selection_performance's
# efficiency plots and their MIN_TRUE_ENERGY_MEV denominator floor go down to
# 0 MeV to match (see that constant's own comment).
# Matching radii for the 1-to-1 pairing, identical to
# Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb. Needed again now that
# the NumuCCQuality plot splits the signal by the completeness and purity of
# each cluster's match -- change one of these and that split moves.
radius_completeness       = 2
radius_purity_xz          = 3
radius_purity_yz          = 5
radius_purity_xy          = 5

# HOW MANY of the three projected distances a reco point must satisfy to count
# as matched. 3 = the historic all-three rule; 2 = any two, so no single cut can
# veto a match on its own.
#
# With 2 and radius_purity_xz raised 2 -> 3, measured over the full sample
# (553 pairs, 1.29M points): ~43,000 more points accepted, 16 pairs promoted
# into the high-signal region, and only 2 of 553 pairings repoint to a
# different true cluster -- so completeness moves very little. See
# Purity_Cut_Geometry/ for the study.
purity_min_projections    = 2
min_recopoints_threshold  = 5

min_cluster_energy        = 100     # NOT APPLIED (Apply_energy_cutoff = False below) -- kept only as
                                     # a record of the value the cluster-level cut used to use
min_true_point_energy     = 0.02    # MeV per POINT (Apply_trueenergy_pointwise_cutoff below)
min_true_points_cutoff    = 200     # NOT APPLIED (Apply_min_true_points_cutoff = False below)

Apply_energy_cutoff                         = False   # per-cluster 100 MeV floor REMOVED; only
                                                        # the per-point 0.02 MeV cut below still applies
Apply_trueenergy_pointwise_cutoff           = True    # drop true POINTS below min_true_point_energy
Apply_min_true_points_cutoff                = False
Apply_wire_readout_sensitive_xz_plane_cut   = True
Apply_time_window_cut                       = False   # must stay disabled -- no per-point true time in this format
# The dead-area cut is APPLIED, just not here: PARENT_DIR above is the tree
# preprocess_deadarea_cut.py already cut. Set this True only if you point
# PARENT_DIR back at a raw tree.
Apply_deadarea_cut                          = False

# COSMIC TAGGER CUT: drop in-beam activity the WireCell cosmic taggers flagged.
# Applied to the AfterBeamWindowCut selection only -- it is a statement about
# in-beam activity, and there is no in-beam set before that cut.
#
# PER-FLASH (selections.COSMIC_TAG_PROPAGATE_SCOPE = 'flash'): the tagged
# cluster and anything sharing its flash are removed, so a true neutrino at a
# DIFFERENT flash time in the same event SURVIVES. Measured on chunk0:
# 35 of 104 in-beam clusters removed over 33 of 149 events, and 3 of the removed
# clusters each hold more than 90% of a true neutrino.
Apply_cosmic_tagger_cut                     = False

# FIDUCIAL volume (unit: cm) -- the SIGNAL definition. These are the bounds
# for the vertex_in_volume flag, and that is ALL they are: an interaction
# counts as signal when its VERTEX sits inside this volume.
#
# They are deliberately NOT applied to the true points. The points keep
# every deposit the wire-readout cut left, because a neutrino that starts
# inside the fiducial volume and throws tracks past its edge is still that
# neutrino, and the energy it put outside still belongs to it. Cutting the
# points here would quietly shrink the true energy of exactly the
# interactions nearest the boundary.
#
# Taken from selections.py rather than written out here so the signal
# definition lives in one place.
from selections import (Fiducial_X_MIN, Fiducial_X_MAX,
                        Fiducial_Y_MIN, Fiducial_Y_MAX,
                        Fiducial_Z_MIN, Fiducial_Z_MAX)
x_min, x_max = Fiducial_X_MIN, Fiducial_X_MAX
y_min, y_max = Fiducial_Y_MIN, Fiducial_Y_MAX
z_min, z_max = Fiducial_Z_MIN, Fiducial_Z_MAX

# Metadata label only -- there is no 2-view/3-view distinction in the
# charge-light format, so this is just a constant.
view = "combined"

# ========================================================================
# RECO OVERLAY
# ========================================================================
# One plot is drawn PER RECO SELECTION. The stack is identical in all of them --
# the true side is not touched -- so what each plot compares is the same true
# population against a differently-cut reco population.
#
# The label goes in the plot title, the legend and BOTH output filenames, so the
# versions sit side by side in one directory instead of overwriting each other.
#
# KEEP THE LABELS HONEST: nothing checks that a label matches the cuts the main
# loop actually applies, and a plot labelled with the wrong selection is worse
# than no plot. Adding a selection means adding it here AND building its cluster
# dict in the loop below.
RECO_SELECTION_NOCUTS = DEFAULT_RECO_CUTS_LABEL       # 'NoCuts': every reco cluster
RECO_SELECTION_BEAM   = 'AfterBeamWindowCut'          # flash time inside the beam window
RECO_SELECTION_LABELS = [RECO_SELECTION_NOCUTS, RECO_SELECTION_BEAM]

# ========================================================================
# SAVE PER-CLUSTER XZ / YZ / XY VIEWS?
# ========================================================================
# One figure per saved cluster, three panels each, drawn from full point
# clouds -- much slower per figure than any histogram here, and it writes one
# directory per event that contributes. Off by default; turn it on when you
# want to look at what a region of the completeness-purity plane contains.
#
# At most PAIRS_PER_CELL per 10%% x 10%% cell (100 cells), so the count is
# bounded however long the job runs.
# WHICH TRUTH FILE the true clusters come from.
#
#   True  -> sed-sce_smear_readout : true positions WITH the space-charge
#            displacement, which is what clustering-global's reco positions
#            carry. Matching 225,633 reco points to their nearest true point
#            over 8 events gives a median residual of 0.394 cm against this
#            file and 0.555 cm against sed-smear, so this is the variant the
#            reco actually sits on.
#   False -> sed-smear_readout : true positions, no space charge. What every
#            run before 2026-08-14 used.
#
# Only x/y/z differ between the two -- every energy, cluster id and nu_idx is
# identical -- so this moves completeness and purity and nothing else.
TRUE_SOURCE_SCE     = True

# WHICH ID FIELD defines a reco cluster in clustering-global.
#
#   'cluster_id'      -> the COARSE grouping: everything the charge-light
#            matching tied to one flash counts as ONE reco cluster. Measured
#            over 1363 events, 50 of 51 groups of beam-window clusters that
#            share a flash time sit inside a single cluster_id, so this is
#            very nearly 'group all in-beam activity together'.
#   'real_cluster_id' -> the FINER grouping, used by every run before
#            2026-08-15. It keeps apart pieces that cluster_id merges,
#            including, in 13 of those 51 groups, two genuinely different
#            neutrinos.
#
# The two fields are identical in img-global and in the truth files; the split
# exists only in clustering-global.
RECO_ID_FIELD       = 'cluster_id'

SAVE_CLUSTER_VIEWS  = True

# ========================================================================
# OUTPUTS THAT ONLY CHANGE WITH THE INPUT SAMPLE
# ========================================================================
# These describe the SAMPLE, not the reconstruction or selection code under
# study, so a re-run on the same input redraws pictures an earlier job already
# produced -- for no new information and a noticeable share of the runtime.
# They stay off until the input sample changes; the code that draws them is
# untouched behind this switch, so setting it True is the whole cost of getting
# them back.
#
#   signal_neutrino_multiplicity/    interactions per event
#   energy_reconstruction/           true vs reco energy of well matched pairs
#   Saved_Clusters/                  the completeness-purity grid of pair views
#   two_neutrino_in_beam/            events holding two selected neutrinos
#
# NOT covered, and still drawn every run: the unselected nue CC views, which are
# few and are about the reconstruction rather than the sample. So is the index
# that lists them, written to selection_completeness_vs_purity/.
#
# TWO POPULATIONS LEFT THIS NOTEBOOK ENTIRELY, each to its own notebook beside
# the contaminated pairs -- a per-cluster picture is what those are for, while
# this notebook draws distributions:
#
#   below-60%-completeness pairs -> DrawRecoTrueClusters_Below_60pc_Completeness.ipynb
#   selected cosmic clusters     -> Draw_Selection_Cosmics.ipynb
#   reco cluster flashes         -> Draw_Cluster_Flashes.ipynb
#   cosmic-tagger results        -> Draw_TaggedCosmics.ipynb
#   true-cluster stacks          -> Draw_Signal_Background_True.ipynb
b_redraw_established_outputs = False

# ========================================================================
# WHICH EFFICIENCY FIGURES TO DRAW
# ========================================================================
# selection_efficiency IS the runtime of this job. The same 43 figures per
# channel are drawn once per (bin width x binning x uncertainty style) = 18
# copies, giving 2088 figures at ~0.14 s each -- 287 s of a 415 s run, against
# 127 s for the event loop and ~15 s for every other plot combined. Measured
# per directory from file mtimes; the three binnings and the three styles cost
# exactly the same, 696 figures and ~95 s each.
#
# So the grid is cut down to one cell of the nine. What is lost is VIEWS, not
# numbers: efficiency.txt, selection_performance_info and the ROOT histograms
# are all built from the unbinned efficiencies and do not change. Widen either
# tuple back out for a study that needs the other renderings -- one binning and
# one style cost ~32 s, all nine cost ~287 s, and everything between is linear.
#
# Names must come from EFFICIENCY_BINNINGS and UNCERTAINTY_STYLES in
# draw_selection_performance.py; a typo raises below rather than silently
# drawing nothing.
EFFICIENCY_BINNINGS_DRAWN = ('tail_1bin_above_1000MeV',)
UNCERTAINTY_STYLES_DRAWN  = ('band',)

EFFICIENCY_BINNINGS_SELECTED = tuple(b for b in EFFICIENCY_BINNINGS
                                     if b[0] in EFFICIENCY_BINNINGS_DRAWN)
UNCERTAINTY_STYLES_SELECTED  = tuple(u for u in UNCERTAINTY_STYLES
                                     if u in UNCERTAINTY_STYLES_DRAWN)
for name in EFFICIENCY_BINNINGS_DRAWN:
    if name not in [b[0] for b in EFFICIENCY_BINNINGS]:
        raise ValueError(f"unknown efficiency binning {name!r}; choose from "
                         f"{[b[0] for b in EFFICIENCY_BINNINGS]}")
for name in UNCERTAINTY_STYLES_DRAWN:
    if name not in UNCERTAINTY_STYLES:
        raise ValueError(f"unknown uncertainty style {name!r}; choose from "
                         f"{list(UNCERTAINTY_STYLES)}")

# The SELECTION PERFORMANCE plots need the reco-true pairing: every one of them
# is built on it. Nothing else in this notebook does, now that the true-cluster
# stacks (the other user of PLOT_VARIANTS, and of the pairing when the quality
# split was on) live in Draw_Signal_Background_True.ipynb -- so the two switches
# have collapsed into one.
b_draw_selection_performance = True

NEEDS_PAIRING = b_draw_selection_performance

print("\nCuts applied (true side):")
if Apply_energy_cutoff:
    print(f"- Energy cutoff applied AFTER the volume cuts (threshold {min_cluster_energy} MeV "
          f"of SURVIVING energy, using sed-smear's per-point 'e' field)")
if Apply_trueenergy_pointwise_cutoff:
    print(f"- True POINT-wise energy cutoff applied BEFORE it (threshold {min_true_point_energy} MeV per point)")
if Apply_wire_readout_sensitive_xz_plane_cut:
    print(f"- Wire readout sensitive xz plane cut applied")
if Apply_deadarea_cut:
    print(f"- Dead area cut applied HERE")
else:
    print(f"- Dead area cut applied UPSTREAM by preprocess_deadarea_cut.py (baked into PARENT_DIR)")
print(f"- FIDUCIAL bounds (vertex_in_volume only -- NOT a cut on the true points): x [{x_min}, {x_max}], y [{y_min}, {y_max}], z [{z_min}, {z_max}] cm")

print(f"\nReco selections counted (not plotted): " + ", ".join(RECO_SELECTION_LABELS))
print(f"  beam window: {BEAM_WINDOW_MIN_US} - {BEAM_WINDOW_MAX_US} us (flash time)")
print(f"  reco energy = {RECO_WORK_FUNCTION_EV} eV * charge / {RECO_RECOMBINATION_FACTOR}")

print("\nStack components (bottom first):")
for component in SIGNAL_BACKGROUND_COMPONENTS:
    print(f"- {component['key']}")

# ========================================================================
# ONE-TIME EXTRACTION
# ========================================================================
# Both paths are idempotent -- a data/<k>/ (nuecc) or data/ (tagger) that already
# holds files is left alone -- so re-running this notebook never re-extracts.
# staging_seconds is reported in summary.txt; ~0 on a re-run and on the tagger
# sample (data/ already present).
staging_start = time.time()
if SAMPLE == "nuecc":
    staged_chunks = []
    for source_chunk in NUECC_SOURCE_CHUNKS:
        print(f"Staging {NUECC_N_FILES} zips from {NUECC_BEE_ROOT / source_chunk} "
              f"in subchunks of {NUECC_CHUNK_SIZE}")
        staged_chunks += stage_nuecc_chunks(
            NUECC_BEE_ROOT / source_chunk, NUECC_STAGING_ROOT,
            n_files=NUECC_N_FILES, chunk_size=NUECC_CHUNK_SIZE)
    print(f"Staged {len(staged_chunks)} subchunk dir(s) from "
          f"{len(NUECC_SOURCE_CHUNKS)} source chunk(s)")
elif PARENT_DIR.exists():
    for subdir in sorted(PARENT_DIR.iterdir()):
        if subdir.is_dir():
            ensure_data_extracted(subdir)
else:
    print(f"Error: Parent directory {PARENT_DIR} does not exist")
staging_seconds = time.time() - staging_start


In [ ]:
def find_all_input_directories(parent_dir):
    """
    Scan parent directory for all subdirectories containing 'data' folder.
    Returns a list of file directories (file0/, file1/, etc.).
    """
    parent_dir = Path(parent_dir)
    if not parent_dir.exists():
        print(f"Error: Parent directory {parent_dir} does not exist")
        return []

    data_dirs = []
    for subdir in sorted(parent_dir.iterdir()):
        if subdir.is_dir():
            data_path = subdir / "data"
            if data_path.exists() and data_path.is_dir():
                data_dirs.append(subdir)
                print(f"Found: {subdir}")

    return data_dirs


def file_index_from_name(name):
    """
    Trailing integer of a file directory name ("file10" -> 10), or None if it
    has no trailing digits. Used by target_file_range so files are selected by
    their real index rather than by position in the lexicographically sorted
    list (file0, file1, file10, file11, file2, ...).
    """
    digits = ""
    for ch in reversed(name):
        if not ch.isdigit():
            break
        digits = ch + digits
    return int(digits) if digits else None


def detect_events_in_directory(input_dir):
    """
    Auto-detect the number of events in a directory.
    Events are identified as numeric subdirectories in data/.
    Returns a sorted list of event numbers.
    """
    input_dir = Path(input_dir)
    data_dir = input_dir / "data"

    if not data_dir.exists():
        print(f"Warning: Data directory {data_dir} does not exist")
        return []

    events = []
    for item in data_dir.iterdir():
        if item.is_dir():
            try:
                events.append(int(item.name))
            except ValueError:
                pass

    return sorted(events)


# Auto-detect all input directories from parent directory
print(f"Scanning parent directory: {PARENT_DIR}")
print("-" * 60)
input_directories = find_all_input_directories(PARENT_DIR)
if SAMPLE == "nuecc":
    # staging/ can still hold chunk dirs from an earlier run with a larger
    # NUECC_N_FILES -- keep only the ones stage_nuecc_chunks() just wrote.
    staged_names = {d.name for d in staged_chunks}
    input_directories = [d for d in input_directories if d.name in staged_names]
if num_files_to_process is not None:
    input_directories = input_directories[:num_files_to_process]
else:
    num_files_to_process = len(input_directories)
print("-" * 60)

print(f"\nFound {len(input_directories)} input directories with data/\n")
if input_directories:
    for input_dir in input_directories:
        detected_events = detect_events_in_directory(input_dir)
        if detected_events:
            print(f"  {input_dir.name}/data/: {len(detected_events)} events ({min(detected_events)}-{max(detected_events)})")
        else:
            print(f"  {input_dir.name}/data/: No events found")
else:
    print(f"Error: No subdirectories with 'data/' found in {PARENT_DIR}")


In [ ]:
# ============================================================================
# MAIN PROCESSING LOOP -- combined-APA signal/background distributions
# ============================================================================
# The true-side selection chain below is Reco_Distributions.ipynb's, unchanged:
# sed-smear_readout grouped by REAL_CLUSTER_ID and reassigned to 99990+nu_idx
# (neutrino, one cluster per interaction) / avg-X (cosmic), then the energy and
# fiducial cuts. The reco half of that loop is deliberately absent -- see the
# header.
#
# Only the JOB-LEVEL stack is drawn (see the header): the loops below collect
# clusters, they do not write per-file or per-event output.
#
# Each event contributes two things to the stack: the true cluster variable
# records (which carry total_energy and vertex_in_volume) and the mc.json vertex
# records (which carry interaction_channel). attach_interaction_channel joins the
# second onto the first by (event, cluster id), and the component selectors in
# draw_signal_background.py read both fields off the finished records.

timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = PLOTBASEDIR / f"combined_apa_{timestamp}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\n{'='*70}")
print(f"Output directory: {output_dir}")
print(f"{'='*70}\n")

job_true_var_records     = []   # one record per selected true cluster, channel attached

# Reco cluster selection flow: how many reco clusters survive each stage. Same
# two stages SelectionAnalysis.ipynb uses, but counted in whichever namespace
# RECO_ID_FIELD selects -- with 'cluster_id' the totals are the coarse clusters
# this notebook actually analyses, not the finer real_cluster_id ones.
RECO_FLOW_STAGES = [(RECO_SELECTION_NOCUTS, 'No cuts'),
                    (RECO_SELECTION_BEAM,   '+ Beam window')]
job_reco_flow_counts = {label: 0 for label, _ in RECO_FLOW_STAGES}
job_reco_var_records     = {label: [] for label in RECO_SELECTION_LABELS}  # counted, and the beam-window set feeds the pairing
# Fresh per job so a re-run fills the same view slots again rather than silently
# drawing nothing the second time.
cluster_view_sampler = ClusterViewSampler() if SAVE_CLUSTER_VIEWS else None
cluster_view_root    = output_dir / "job_summary" / "Saved_Clusters"
# The two-neutrino events and the unselected nue CC views are not cells of the
# completeness-purity grid, so they are written beside Saved_Clusters rather than
# inside it, directly under job_summary.
job_view_root        = output_dir / "job_summary"
job_selection_records    = []   # one per selected reco cluster, with its category
job_cluster_type_records = []   # per-true-cluster is_neutrino (build_true_cluster_type_records)
job_vertex_records       = []   # per true neutrino interaction (build_neutrino_vertex_records)
# What the cosmic tagger cut removed, for summary.txt: it is a large cut and
# a job that silently dropped a third of its in-beam clusters should say so.
n_events_cosmic_tagged   = 0
n_clusters_cosmic_tagged = 0
total_events_processed   = 0
total_files_processed    = 0

for file_idx, input_dir in enumerate(input_directories):
    input_file_name = input_dir.name

    # SELECTIVE FILTERING: Skip files that don't match target_file
    if target_file is not None and input_file_name != target_file:
        print(f"Skipping {input_file_name} (target: {target_file})")
        continue

    # SELECTIVE FILTERING: Skip files outside target_file_range (inclusive both
    # ends, matched on the directory name's trailing index -- see
    # file_index_from_name). Applied on top of target_file, not instead of it.
    if target_file_range is not None:
        file_idx = file_index_from_name(input_file_name)
        range_low, range_high = target_file_range
        if file_idx is None or not (range_low <= file_idx <= range_high):
            print(f"Skipping {input_file_name} (target range: file{range_low}..file{range_high})")
            continue

    print(f"\n{'='*70}")
    print(f"FILE {file_idx+1}/{len(input_directories)}: {input_dir}")
    print(f"{'='*70}")

    events_list = detect_events_in_directory(input_dir)
    if not events_list:
        print(f"No events found in {input_dir}, skipping...")
        continue

    event_low = min(events_list)
    event_high = max(events_list) + 1 if num_events_to_process is None else event_low + num_events_to_process

    print(f"Processing events {event_low} to {event_high-1}\n")
    total_files_processed += 1

    # Start of event loop
    for evt in range(event_low, event_high):
        # SELECTIVE FILTERING: Skip events that don't match target_event
        if target_event is not None and evt != target_event:
            continue

        # SELECTIVE FILTERING: Skip events outside target_event_range (inclusive
        # both ends). Applied on top of target_event, not instead of it.
        if target_event_range is not None:
            event_range_low, event_range_high = target_event_range
            if not (event_range_low <= evt <= event_range_high):
                continue

        result = read_charge_light_files_for_event(input_dir, evt)
        if result is None:
            print(f"  Event {evt}: could not read data, skipping")
            continue

        event_key = f"{input_file_name}_{evt}"

        true_key = 'true_clustering_sce' if TRUE_SOURCE_SCE else 'true_clustering'
        if result.get(true_key) is None:
            raise RuntimeError(f"event {event_key}: {true_key} missing -- "
                               f"sed-sce_smear_readout.json is absent for this event")
        x_true, y_true, z_true, id_true, q_true, real_id_true, e_true, nu_idx_true = result[true_key]
        x_clu,  y_clu,  z_clu,  id_clu,  q_clu,  real_id_clu                       = result['clustering']
        mc_tree = result['mc']
        op_data = result['op']

        # ------------------------------------------------------------------
        # TRUE POINTS: the chosen truth variant in the standard 7-column shape
        # (energy = per-point 'e' in MeV; q_true = 'nu_idx', 0=cosmic,
        # 1/2/...=which neutrino interaction), reassigned to 99990+nu_idx
        # (neutrino) / avg-X (cosmic), then cut.
        # real_id_true (real_cluster_id), NOT id_true: cluster_id is a coarser
        # grouping that can merge physically distinct tracks.
        # ------------------------------------------------------------------
        true_points = build_true_points_charge_light(
            x_true, y_true, z_true, real_id_true, q_true, energy=e_true, nu_idx=nu_idx_true)
        true_points = reassign_cluster_ID_true_charge_light(true_points)

        # Snapshot BEFORE the cuts: build_neutrino_vertex_records uses it to say
        # what a removed neutrino actually deposited. Grouping only, no filtering.
        clusters_true_precut = GroupClustersByID(true_points)

        # CUT ORDER: every POINT-level cut runs before every CLUSTER-level one.
        #
        # The fiducial and dead-area cuts delete individual points; the energy
        # and min-point cuts test a whole cluster and keep or drop it. Running a
        # cluster-level test first means testing a quantity the later point-level
        # cuts then change -- and that is not hypothetical: with the energy cut
        # first, a cluster admitted at 116 MeV came out of the fiducial cut with
        # 64 MeV of surviving points and was plotted BELOW the 100 MeV threshold
        # it had supposedly passed. Out-of-volume neutrinos showed it worst, most
        # of their deposit being outside the volume by construction.
        #
        # In this order min_cluster_energy means 100 MeV of energy that SURVIVED
        # the cuts -- the same quantity the histograms are filled with -- so
        # nothing can appear below the threshold.
        if Apply_wire_readout_sensitive_xz_plane_cut:
            true_points = apply_wire_readout_sensitive_yz_plane_cut_true(true_points)
        if Apply_deadarea_cut:
            # The only thing that writes per-event output, and only when the cut
            # is run HERE rather than upstream -- so its directory is created
            # here rather than for every event of every run.
            deadarea_dir = output_dir / input_file_name / f"event_{evt:03d}"
            deadarea_dir.mkdir(parents=True, exist_ok=True)
            true_points = apply_deadarea_cut_true_charge_light(true_points, output_dir=deadarea_dir, event=evt, file_name=input_file_name)
        if len(true_points) == 0:
            print(f"  Event {evt}: no true points survive the volume cuts, skipping")
            continue
        # POINT-wise first, so the cluster total the cluster cut tests is the
        # total of the points that survive.
        if Apply_trueenergy_pointwise_cutoff:
            true_points = apply_true_pointwise_energy_cutoff(true_points, min_true_point_energy)
        if Apply_energy_cutoff:
            true_points = apply_energy_cutoff(true_points, min_cluster_energy)
        if Apply_min_true_points_cutoff:
            true_points = apply_min_true_points_cutoff(true_points, min_true_points_cutoff)

        if len(true_points) == 0:
            print(f"  Event {evt}: no true points remain after cuts, skipping")
            continue

        clusters_true = GroupClustersByID(true_points)

        # ------------------------------------------------------------------
        # TRUTH LABELS: neutrino/cosmic per cluster, and the mc.json interaction
        # vertices joined to their true cluster by nu_idx (cluster_id =
        # 99990+nu_idx, an exact key). vertex_in_volume uses the same bounds as
        # the fiducial cut; interaction_channel is numu_CC / nue_CC / NC.
        # ------------------------------------------------------------------
        event_cluster_type_records = build_true_cluster_type_records(
            clusters_true, input_file_name, evt, event_key)
        event_vertex_records = build_neutrino_vertex_records(
            flatten_mc_tree(mc_tree), clusters_true, input_file_name, evt, event_key,
            x_min=x_min, x_max=x_max, y_min=y_min, y_max=y_max, z_min=z_min, z_max=z_max,
            clusters_true_precut=clusters_true_precut, min_cluster_energy=min_cluster_energy)

        # ------------------------------------------------------------------
        # CLUSTER RECORDS + DRAWING (draw_signal_background.py)
        # ------------------------------------------------------------------
        event_true_var_records = build_true_cluster_variable_records(
            clusters_true, input_file_name, evt, event_key, "Combined",
            vertex_records=event_vertex_records)
        attach_interaction_channel(event_true_var_records, event_vertex_records)

        # ------------------------------------------------------------------
        # RECO CLUSTERS. Not plotted -- counted for summary.txt, and the
        # beam-window set is what the 1-to-1 pairing runs against.
        # clustering-global (post charge-light matching) grouped by
        # REAL_CLUSTER_ID, and nothing else: no beam-window cut, no fiducial
        # cut, no minimum point count. That is what RECO_CUTS_LABEL = 'NoCuts'
        # names, and adding a cut here means changing that label too.
        #
        # real_cluster_id, not cluster_id: cluster_id is a coarser grouping that
        # can merge physically distinct tracks (same reasoning as the true side).
        # reassign_cluster_ID_reco is deliberately NOT called -- it relabels
        # clusters by rounded average X, which MERGES clusters that happen to
        # share one, and this plot counts reco clusters.
        # ------------------------------------------------------------------
        reco_ids_clu = id_clu if RECO_ID_FIELD == 'cluster_id' else real_id_clu
        predicted_points = np.column_stack((x_clu, y_clu, z_clu, reco_ids_clu, q_clu))

        # BEAM-WINDOW CUT, the second selection. op.json flashes are attached to
        # img-global clusters and then bridged onto clustering-global clusters by
        # point charge; a reco cluster passes if its bridged flash time falls in
        # [BEAM_WINDOW_MIN_US, BEAM_WINDOW_MAX_US]. Identical to the cut
        # Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb applies, and it
        # reads the same two constants, so widening the window there widens it here.
        #
        # build_img_cluster_flash_metadata keys its records by REAL_CLUSTER_ID
        # (deliberately -- see metadata.py). When RECO_ID_FIELD is 'cluster_id'
        # the points are grouped in a DIFFERENT namespace, so the beam-window ids
        # must be translated or np.isin below matches nothing and every event
        # comes out empty. A cluster_id passes if any of its real sub-clusters
        # has a beam-window flash, which is what treating them as one activity
        # means.
        #
        # A cluster with NO bridged flash is CUT: it has no time, so it cannot be
        # shown to be in the beam window. That is the evaluation's behaviour too.
        event_flash_metadata_list = build_cluster_flash_metadata(
            op_data, input_file_name, evt, "Combined", event_key)
        event_img_cluster_flash_records = build_img_cluster_flash_metadata(
            result['reco'], result['clustering'], event_flash_metadata_list,
            input_file_name, evt, "Combined", event_key)
        clu_beam_window_ids = {float(r['clustering_cluster_id']) for r in event_img_cluster_flash_records
                               if BEAM_WINDOW_MIN_US <= r['flash_time'] <= BEAM_WINDOW_MAX_US}
        real_to_coarse = {float(r): float(c) for r, c in zip(real_id_clu, id_clu)}
        if RECO_ID_FIELD == 'cluster_id':
            clu_beam_window_ids = {real_to_coarse[r] for r in clu_beam_window_ids
                                   if r in real_to_coarse}

        if len(predicted_points) and clu_beam_window_ids:
            beam_ids_array = np.fromiter(clu_beam_window_ids, dtype=float, count=len(clu_beam_window_ids))
            predicted_points_beam = predicted_points[np.isin(predicted_points[:, 3], beam_ids_array)]
        else:
            predicted_points_beam = predicted_points[:0]

        # COSMIC TAGGER CUT, on the beam-window survivors and nothing else.
        #
        # PER-FLASH: the tagged cluster and anything sharing its flash are
        # removed; activity at other flash times is kept. Set
        # selections.COSMIC_TAG_PROPAGATE_SCOPE='event' for all-or-nothing.
        # interaction and the grouping is flash-based, so keeping one cluster
        # while cutting its neighbour would assert a distinction the
        # reconstruction cannot make.
        #
        # This REMOVES TRUE NEUTRINOS sharing a window with a tagged cosmic. That
        # loss is real and accepted -- see apply_cosmic_tagger_cut in selections.py.
        if Apply_cosmic_tagger_cut:
            predicted_points_beam, tagger_cut_info = apply_cosmic_tagger_cut(
                predicted_points_beam, result.get('taggers'))
            if tagger_cut_info['n_tagged_clusters']:
                n_events_cosmic_tagged += 1
                n_clusters_cosmic_tagged += tagger_cut_info['n_tagged_clusters']
                # Drawn from the PRE-cut clusters -- the removed ones are, by
                # construction, no longer in the post-cut set.
                print(f"    cosmic tagger cut: removed "
                      f"{tagger_cut_info['n_tagged_clusters']} in-beam cluster(s) "
                      f"({tagger_cut_info['n_direct']} tagged directly by "
                      f"{', '.join(tagger_cut_info['tagged_by'])}), "
                      f"{tagger_cut_info['n_points_removed']} points")

        event_reco_points_by_selection = {
            RECO_SELECTION_NOCUTS: predicted_points,
            RECO_SELECTION_BEAM:   predicted_points_beam,
        }
        event_reco_var_records = {}
        event_reco_clusters    = {}
        for selection_label, selection_points in event_reco_points_by_selection.items():
            selection_clusters = GroupClustersByID(selection_points) if len(selection_points) else {}
            event_reco_clusters[selection_label] = selection_clusters
            event_reco_var_records[selection_label] = build_reco_cluster_variable_records(
                selection_clusters, input_file_name, evt, event_key, "Combined")

        # ------------------------------------------------------------------
        # 1-TO-1 TRUE-RECO PAIRING, for the NumuCCQuality split only.
        # Completeness and purity are computed here because that split needs a
        # per-cluster quality, and MatchTrueToReco1to1 needs both to choose a
        # pair. Nothing else on any of these plots depends on the pairing.
        #
        # Paired against the BEAM-WINDOW reco clusters, not the uncut ones: that
        # is the population the plot showing this split is drawn on, and it is
        # the same pairing Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb
        # performs, with the same radii. Pairing against the uncut set would
        # score each true cluster against reco clusters the plot does not show.
        # ------------------------------------------------------------------
        event_pair_metadata_list = []
        completeness_results = purity_results = []
        if NEEDS_PAIRING:
         clusters_reco_for_pairing = event_reco_clusters[RECO_SELECTION_BEAM]
         cluster_category_results = cluster_category(
             clusters_true, output_dir=None, event=evt, apa="Combined", file_name=input_file_name)
         completeness_results = EvaluateCompleteness(
             clusters_true, clusters_reco_for_pairing, event_key,
             radius_completeness, min_recopoints_threshold)
         purity_results = EvaluatePurity(
             clusters_true, clusters_reco_for_pairing, event_key,
             radius_purity_xz, radius_purity_yz, radius_purity_xy,
             min_projections=purity_min_projections)
         event_matched_pairs = MatchTrueToReco1to1(completeness_results, purity_results)
         event_pair_metadata_list = add_metadata_true_reco_pair_cluster(
             event_matched_pairs, cluster_category_results,
             file_name=input_file_name, event=evt, apa="Combined", view=view, event_key=event_key)
        # None for both metrics when the pairing did not run, which is what the
        # quality selectors already treat as "not well reconstructed".
        attach_pair_metrics(event_true_var_records, event_pair_metadata_list)

        # ------------------------------------------------------------------
        # SELECTION PERFORMANCE: label every selected reco cluster.
        # Categorised against the SAME reco selection the pairing used, so a
        # cluster is never labelled by an overlap the plots do not show.
        # ------------------------------------------------------------------
        if b_draw_selection_performance:
            event_selection_records = categorize_reco_clusters(
                event_reco_var_records[RECO_SELECTION_BEAM],
                purity_results, completeness_results, event_true_var_records)
            job_selection_records.extend(event_selection_records)

            # XZ/YZ/XY views, drawn HERE because this is the only place the point
            # clouds exist -- they are far too large to carry to job level. The
            # sampler keeps at most one pair per cell, so most events draw nothing.
            if cluster_view_sampler is not None:
                save_event_cluster_views(
                    event_selection_records, clusters_true,
                    event_reco_clusters[RECO_SELECTION_BEAM],
                    cluster_view_sampler, cluster_view_root, event_key,
                    job_root=job_view_root,
                    draw_pair_cells=b_redraw_established_outputs,
                    draw_two_neutrino=b_redraw_established_outputs)

        # ------------------------------------------------------------------
        # AGGREGATE TO JOB LEVEL
        # ------------------------------------------------------------------
        job_true_var_records.extend(event_true_var_records)
        for selection_label in RECO_SELECTION_LABELS:
            job_reco_var_records[selection_label].extend(event_reco_var_records[selection_label])
        for selection_label, _stage in RECO_FLOW_STAGES:
            job_reco_flow_counts[selection_label] += len(event_reco_clusters[selection_label])
        job_cluster_type_records.extend(event_cluster_type_records)
        job_vertex_records.extend(event_vertex_records)

        n_true_neutrino = sum(1 for r in event_true_var_records if r['is_neutrino'])
        n_numu_cc_in    = sum(1 for r in event_true_var_records
                              if r.get('interaction_channel') == 'numu_CC'
                              and r.get('vertex_in_volume') is True)
        print(
            f"  Event {evt}: "
            f"true clusters={len(event_true_var_records)} (neutrino={n_true_neutrino}), "
            f"numu CC in volume={n_numu_cc_in}, "
            f"reco clusters={len(event_reco_var_records[RECO_SELECTION_NOCUTS])} "
            f"(in beam window={len(event_reco_var_records[RECO_SELECTION_BEAM])}), "
            f"1-to-1 pairs={len(event_pair_metadata_list)}, "
            f"neutrino interactions in mc={len(event_vertex_records)}"
        )
        total_events_processed += 1

    # Per-unit status, so a long multi-chunk job shows a milestone as each staged
    # chunk__subchunk finishes -- and a louder marker when a whole source chunk
    # (10 units) is done -- rather than only a scroll of per-event lines.
    _src_chunk = input_file_name.split("__")[0]
    print(f"  ==> unit {file_idx + 1}/{len(input_directories)} done: {input_file_name}  "
          f"({total_events_processed} events, {len(job_true_var_records)} true clusters, "
          f"{(time.time() - job_start_time) / 60:.1f} min elapsed)")
    _next = input_directories[file_idx + 1].name if file_idx + 1 < len(input_directories) else ""
    if _src_chunk and not _next.startswith(_src_chunk + "__"):
        print(f"  ===================  SOURCE CHUNK {_src_chunk} COMPLETE  "
              f"({total_events_processed} events so far)  ===================")

# ============================================================================
# JOB-LEVEL STACK: every file and event
# ============================================================================
print(f"\n{'='*70}")
print(f"JOB SUMMARY: {total_files_processed} file(s), {total_events_processed} event(s) processed")
print(f"Total selected true clusters: {len(job_true_var_records)}")
print(f"{'='*70}")

job_output_dir = output_dir / "job_summary"
job_output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# SELECTION PERFORMANCE -- the three reco-space plots
# ============================================================================
if b_draw_selection_performance and job_selection_records:
    print(f"\n  SELECTION PERFORMANCE ({len(job_selection_records)} selected reco clusters)")
    efficiencies_by_width = {}
    for bin_width in BIN_WIDTHS_MEV:
        # Plot 1. Raises if the categories do not partition the selection, so a
        # miscategorised cluster stops the run rather than reaching a figure.
        reco_dir = plot_directory(job_output_dir, 'selection_reco', bin_width)
        # One stack per high-signal threshold. Only the in-volume split moves
        # between them -- cosmic and out-of-volume are decided by the match and
        # the vertex, which no quality threshold touches -- so the stacks
        # show the same total with signal and contaminated trading places.
        for stack_threshold in EFFICIENCY_THRESHOLDS:
            for fill_style in ('mixed', 'solid'):
                by_key, all_energies = draw_reco_selection_stack(
                    job_selection_records, reco_dir, "Job Level", "job", "Combined",
                    bin_width=bin_width, reco_cuts_label=RECO_SELECTION_BEAM,
                    threshold=stack_threshold, fill_style=fill_style)

        # Plot 3 inputs, per channel.
        efficiencies = [build_selection_efficiency(job_selection_records, job_vertex_records,
                                                   channel, bin_width=bin_width)
                        for channel in CHANNELS]
        # The same efficiency at every quality threshold. Rebuilt from each pair's
        # own metrics, so this costs nothing beyond the histogramming.
        by_threshold = {
            channel: {t: build_selection_efficiency(job_selection_records, job_vertex_records,
                                                    channel, bin_width=bin_width, threshold=t)
                      for t in EFFICIENCY_THRESHOLDS}
            for channel in CHANNELS}
        efficiencies_by_width[bin_width] = efficiencies

        # The same efficiency split by how many signal neutrinos the event held.
        # Single- and multi-neutrino events are drawn together, since the question
        # is whether they differ.
        # ...at EVERY signal definition, not just the default one: the split by
        # multiplicity is a statement about the signal, so it has to be available
        # wherever the signal is redefined. Only the numerators change with the
        # threshold, so this is histogramming already-computed metrics again.
        by_multiplicity_threshold = {
            t: {channel: {cls: build_selection_efficiency(
                              job_selection_records, job_vertex_records, channel,
                              bin_width=bin_width, threshold=t, multiplicity_class=cls)
                          for cls in ('single', 'multi', 'all')}
                for channel in CHANNELS}
            for t in EFFICIENCY_THRESHOLDS}
        # The default threshold's version is what efficiency.txt reports.
        by_multiplicity = by_multiplicity_threshold[HIGH_SIGNAL_THRESHOLD]
        # Everything below goes to
        # selection_efficiency/<width>MeV/<channel>/<binning>/<style>/<signal definition>/.
        #
        # EVERY efficiency figure is drawn in three BINNINGS (fine throughout,
        # and two that merge the sparse tail above 1000 MeV) x three
        # UNCERTAINTY styles (none, Clopper-Pearson band, Clopper-Pearson bars)
        # = nine versions, each under the subfolder for its signal definition, in
        # selection_efficiency/<width>MeV/<channel>/<binning>/<style>/<signal definition>/.
        #
        # The rebinning is arithmetic on histograms that already exist, so the
        # extra binnings cost figure-drawing time and nothing else.
        for binning_dir, binning_tag, n_tail_bins in EFFICIENCY_BINNINGS_SELECTED:
            def rebinned(efficiency):
                if n_tail_bins is None:
                    return efficiency
                merged = rebin_efficiency_tail(efficiency, n_tail_bins)
                merged['binning_tag'] = binning_tag
                return merged

            binned_by_multiplicity = {
                t: {c: {k: rebinned(e) for k, e in by_class.items()}
                    for c, by_class in by_channel.items()}
                for t, by_channel in by_multiplicity_threshold.items()}
            binned_by_threshold = {c: {t: rebinned(e) for t, e in thresholds.items()}
                                   for c, thresholds in by_threshold.items()}

            # Per-bin numbers at EVERY threshold, not just the default -- so a
            # comparison at a different signal definition (e.g. completeness_gt_60pc)
            # never needs the job re-run again. Written once per (bin_width,
            # binning_dir), covering every channel, into that binning's own directory.
            write_efficiency_by_threshold_summary(
                binned_by_threshold, plot_directory(job_output_dir, 'selection_efficiency',
                                                     bin_width, None, binning_dir),
                "Job Level", reco_cuts_label=RECO_SELECTION_BEAM)

            for uncertainty in UNCERTAINTY_STYLES_SELECTED:
                variant = UNCERTAINTY_DIRS[uncertainty]
                for mult_threshold, by_channel in binned_by_multiplicity.items():
                    for channel, by_class in by_channel.items():
                        channel_dir = plot_directory(job_output_dir, 'selection_efficiency',
                                                     bin_width, channel, binning_dir, variant)
                        # All Selected does not depend on the threshold, so it is
                        # drawn once (at the default) rather than once per threshold.
                        numerators = (['numerator_high', 'numerator_any']
                                      if mult_threshold == HIGH_SIGNAL_THRESHOLD
                                      else ['numerator_high'])
                        for numerator_key in numerators:
                            for set_label, classes in MULTIPLICITY_FIGURES:
                                draw_efficiency_by_multiplicity(
                                    by_class, channel_dir, "Job Level", "job",
                                    "Combined", numerator_key=numerator_key,
                                    reco_cuts_label=RECO_SELECTION_BEAM,
                                    uncertainty=uncertainty, classes=classes,
                                    set_label=set_label)
                # Per channel: every threshold x {both, high, goodbad} (each in its own
                # signal-definition subfolder), then one figure at the channel level
                # putting all the thresholds together against good+bad.
                for channel, thresholds in binned_by_threshold.items():
                    channel_dir = plot_directory(job_output_dir, 'selection_efficiency',
                                                 bin_width, channel, binning_dir, variant)
                    for efficiency in thresholds.values():
                        for curve_set_label, curves in EFFICIENCY_CURVE_SETS:
                            draw_selection_efficiency(efficiency, channel_dir, "Job Level", "job",
                                                      "Combined", reco_cuts_label=RECO_SELECTION_BEAM,
                                                      curves=curves, curve_set_label=curve_set_label,
                                                      uncertainty=uncertainty)
                    draw_efficiency_threshold_comparison(thresholds, channel_dir, "Job Level", "job",
                                                         "Combined", reco_cuts_label=RECO_SELECTION_BEAM,
                                                         uncertainty=uncertainty)

        write_selection_performance_info(by_key, job_selection_records, efficiencies,
                                         reco_dir, "Job Level", bin_width=bin_width,
                                         reco_cuts_label=RECO_SELECTION_BEAM)
        sel_root = write_selection_performance_root(
            by_key, all_energies, efficiencies, reco_dir, bin_width=bin_width,
            reco_cuts_label=RECO_SELECTION_BEAM,
            filename=f'selection_performance_histograms_{bin_width:.0f}MeV.root')
        counts = {k: len(v) for k, v in by_key.items()}
        print(f"    @{bin_width:.0f}MeV: {counts}")

    # efficiency.txt: the integrated numbers (bin-width independent) plus a
    # per-bin table at the FINEST width drawn, so the integrated value can be
    # checked against the distribution behind it.
    eff_path = write_efficiency_summary(efficiencies_by_width,
                                        plot_directory(job_output_dir, 'selection_efficiency'), "Job Level",
                                        reco_cuts_label=RECO_SELECTION_BEAM,
                                        efficiencies_by_multiplicity=by_multiplicity,
                                        vertex_records=job_vertex_records)
    if b_redraw_established_outputs:
        mult_path = draw_neutrino_multiplicity(
            job_vertex_records, plot_directory(job_output_dir, 'signal_neutrino_multiplicity'),
            "Job Level", "job", "Combined")
        print(f"    neutrino multiplicity: {mult_path.name if mult_path else None}")
    print(f"    efficiency summary: {eff_path.name}")

    if b_redraw_established_outputs:
        # True vs reco energy for well matched pairs, at two quality cuts and two
        # bin widths. Restricted to good pairs because a calibration needs the reco
        # cluster and the true cluster to be the same object.
        energy_dir = plot_directory(job_output_dir, 'energy_reconstruction')
        for quality in ENERGY_RECO_QUALITY:
            for bin_width in BIN_WIDTHS_MEV:
                draw_energy_reconstruction(job_selection_records, job_true_var_records,
                                           energy_dir, "Job Level", "job", "Combined",
                                           bin_width=bin_width, quality=quality,
                                           reco_cuts_label=RECO_SELECTION_BEAM)

    # Reco cluster selection flow, one Total bar per stage.
    DrawRecoClusterSelectionFlow(
        [{'key': label, 'stage': stage, 'geometry': False,
          'total': job_reco_flow_counts[label]} for label, stage in RECO_FLOW_STAGES],
        plot_directory(job_output_dir, 'selection_reco'), "Job Level", "job", "Combined",
        include_geometry_cuts=False)
    print(f"    reco selection flow: "
          f"{', '.join(f'{s}={job_reco_flow_counts[l]}' for l, s in RECO_FLOW_STAGES)}")

    scatter_dir = plot_directory(job_output_dir, 'selection_completeness_vs_purity')
    in_volume_pairs, cosmic_candidates = draw_completeness_vs_purity(
        job_selection_records, scatter_dir, "Job Level", "job", "Combined",
        reco_cuts_label=RECO_SELECTION_BEAM)
    # The same scatter again, with pairs from events holding MORE THAN ONE signal
    # neutrino ringed in red. Counted over the same population the efficiency
    # denominator uses, so "a neutrino" means the same thing in both places.
    multi_neutrino_events = {event for event, n in
                             count_signal_interactions_per_event(job_vertex_records).items()
                             if n > 1}
    draw_completeness_vs_purity(
        job_selection_records, scatter_dir, "Job Level", "job", "Combined",
        reco_cuts_label=RECO_SELECTION_BEAM,
        multi_neutrino_events=multi_neutrino_events)
    # And the same again with ONLY those pairs, so the few red points can be read
    # without the single-neutrino bulk sitting on top of them.
    draw_completeness_vs_purity(
        job_selection_records, scatter_dir, "Job Level", "job", "Combined",
        reco_cuts_label=RECO_SELECTION_BEAM,
        multi_neutrino_events=multi_neutrino_events, multi_neutrino_only=True)
    print(f"    multi-neutrino events: {len(multi_neutrino_events)}")
    # And one per channel. numu CC dominates the combined plot -- 409 of the 553
    # in-volume pairs on the full sample -- so the nue CC and NC populations are
    # only readable on their own axes.
    for channel in CHANNELS:
        draw_completeness_vs_purity(
            job_selection_records, scatter_dir, "Job Level", "job", "Combined",
            reco_cuts_label=RECO_SELECTION_BEAM, channel_only=channel)
    # And one per channel. numu CC dominates the combined plot, so the nue CC
    # and NC populations are only readable on their own axes.
    for channel in CHANNELS:
        draw_completeness_vs_purity(
            job_selection_records, scatter_dir, "Job Level", "job", "Combined",
            reco_cuts_label=RECO_SELECTION_BEAM, channel_only=channel)
    # Both versions: one whose axes start below zero so the cosmics have
    # somewhere to sit, one confined to the physical square. Linear colour scale
    # only -- the log-z version was dropped.
    for with_cosmics in (True, False):
        draw_completeness_vs_purity_colz(
            job_selection_records, scatter_dir, "Job Level", "job", "Combined",
            reco_cuts_label=RECO_SELECTION_BEAM, include_cosmics=with_cosmics,
            log_scale=False)
    print(f"    completeness-vs-purity: {len(in_volume_pairs)} in-volume pairs, "
          f"{len(cosmic_candidates)} cosmic candidates")

    if cluster_view_sampler is not None:
        # The index goes next to the scatter it explains, not inside
        # Saved_Clusters, so it survives a job that draws no grid cells. Its
        # paths are written relative to itself.
        index_path = write_cluster_view_index(
            cluster_view_sampler, cluster_view_root,
            index_root=plot_directory(job_output_dir, 'selection_completeness_vs_purity'),
            job_root=job_view_root)
        print(f"    saved cluster views: {cluster_view_sampler.summary()} -> {index_path.name}")
plt.close('all')

# ============================================================================
# JOB SUMMARY TEXT FILE -- configuration, and how many clusters ended up in each
# stack component (the same selectors the drawer used, so these ARE the
# histogram entry counts).
# ============================================================================
from datetime import timedelta

job_finish_dt = datetime.now()
job_runtime   = time.time() - job_start_time

# The DEFAULT four-band stack's counts. Recomputed rather than taken from the
# drawing loop's last job_selected_by_key: that mapping belongs to whichever
# variant ran last, which is the five-band NumuCCQuality one and has no
# 'numu_CC_in_volume' key at all. The selectors are pure, so recomputing gives
# exactly the counts the NoCuts and AfterBeamWindowCut figures were drawn from.
component_counts = {c['key']: len(c['select'](job_true_var_records))
                    for c in SIGNAL_BACKGROUND_COMPONENTS}
n_job_neutrinos = sum(1 for r in job_true_var_records if r['is_neutrino'])

summary_lines = []
summary_lines.append("=" * 80)
summary_lines.append("JOB SUMMARY -- SIGNAL & BACKGROUND DISTRIBUTIONS")
summary_lines.append("=" * 80)
summary_lines.append(f"Generated: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append("")
summary_lines.append("Configuration:")
summary_lines.append(f"Parent directory: {PARENT_DIR}")
summary_lines.append(f"Plot base directory: {PLOTBASEDIR}")
summary_lines.append(f"Files to process: {files}")
summary_lines.append(f"Events to process: {events}")
# The two choices that decide WHICH points the whole analysis is built from.
# Recorded here because neither is visible in any plot, and both change every
# completeness and purity in the run.
summary_lines.append("")
summary_lines.append("Input definitions:")
summary_lines.append(f"  reco cluster id field:    {RECO_ID_FIELD}"
                     f"   (clustering-global; "
                     f"{'coarse -- all activity on one flash is ONE cluster' if RECO_ID_FIELD == 'cluster_id' else 'fine -- flash-mates stay separate'})")
summary_lines.append(f"  true cluster file:        "
                     f"{'sed-sce_smear_readout   (WITH space charge)' if TRUE_SOURCE_SCE else 'sed-smear_readout   (no space charge)'}")
summary_lines.append(f"  reco cluster file:        clustering-global")
summary_lines.append(f"  purity radii xz/yz/xy:    "
                     f"{radius_purity_xz}/{radius_purity_yz}/{radius_purity_xy} cm, "
                     f"{purity_min_projections} of 3 projections must pass")
if target_file is not None or target_event is not None or target_file_range is not None or target_event_range is not None:
    summary_lines.append(f"Target file: {target_file if target_file else 'all'}")
    summary_lines.append(f"Target event: {target_event if target_event is not None else 'all'}")
    if target_event_range is not None:
        summary_lines.append(f"Target event range: event{target_event_range[0]}..event{target_event_range[1]} (inclusive)")
    if target_file_range is not None:
        summary_lines.append(f"Target file range: file{target_file_range[0]}..file{target_file_range[1]} (inclusive)")
summary_lines.append("")
summary_lines.append("Cuts (the true-side subset of Evaluation_ChargeLightMatching_AfterBeamWindowCut.ipynb):")
summary_lines.append(f"  energy cutoff:            {Apply_energy_cutoff} ({min_cluster_energy} MeV)")
summary_lines.append(f"  point-wise energy cutoff: {Apply_trueenergy_pointwise_cutoff} ({min_true_point_energy} MeV per true point)")
summary_lines.append(f"  min true points cutoff:   {Apply_min_true_points_cutoff} ({min_true_points_cutoff})")
summary_lines.append(f"  wire readout volume cut:  {Apply_wire_readout_sensitive_xz_plane_cut}")
summary_lines.append(f"  dead area cut here:       {Apply_deadarea_cut} (applied upstream when False)")
summary_lines.append(f"  volume bounds: x [{x_min}, {x_max}], y [{y_min}, {y_max}], z [{z_min}, {z_max}] cm")
summary_lines.append(f"  RECO selections (counted, NOT plotted): {', '.join(RECO_SELECTION_LABELS)}")
summary_lines.append(f"    {RECO_SELECTION_NOCUTS}: every reco cluster; no beam-window, "
                     f"fiducial or point-count cut")
summary_lines.append(f"    {RECO_SELECTION_BEAM}: bridged flash time in "
                     f"[{BEAM_WINDOW_MIN_US}, {BEAM_WINDOW_MAX_US}] us; a cluster with no flash is cut")
summary_lines.append(f"  reco energy estimate:     {RECO_WORK_FUNCTION_EV} eV * charge / "
                     f"{RECO_RECOMBINATION_FACTOR} = {reco_cluster_energy_mev(1.0):.6g} MeV per unit charge")
summary_lines.append(f"  1-to-1 pairing:           run against the {RECO_SELECTION_BEAM} reco "
                     f"clusters (radii {radius_completeness}/{radius_purity_xz}/"
                     f"{radius_purity_yz}/{radius_purity_xy}, min reco points {min_recopoints_threshold})")
summary_lines.append("    used ONLY by the NumuCCQuality split; the other plots do not depend on it")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB-LEVEL AGGREGATION")
summary_lines.append("=" * 80)
summary_lines.append(f"Total files processed: {total_files_processed}")
summary_lines.append(f"Total events processed: {total_events_processed}")
summary_lines.append(f"Total selected true clusters: {len(job_true_var_records)}")
summary_lines.append(f"  of which true neutrino clusters: {n_job_neutrinos}")
for selection_label in RECO_SELECTION_LABELS:
    summary_lines.append(f"Total reco clusters, {selection_label} (not plotted): "
                         f"{len(job_reco_var_records[selection_label])}")
summary_lines.append(f"Total true neutrino interactions (mc.json): {len(job_vertex_records)}")
if Apply_cosmic_tagger_cut:
    summary_lines.append(f"Cosmic tagger cut: removed {n_clusters_cosmic_tagged} in-beam "
                         f"cluster(s) over {n_events_cosmic_tagged} event(s) -- per-flash")
    summary_lines.append("  (COSMIC_TAG_PROPAGATE_SCOPE = 'flash'), so a true neutrino at a DIFFERENT")
    summary_lines.append("  flash time is KEPT (see selections.apply_cosmic_tagger_cut)")
    summary_lines.append("  The removed clusters are drawn by Draw_TaggedCosmics.ipynb")
else:
    summary_lines.append("Cosmic tagger cut: NOT applied")
summary_lines.append("")
summary_lines.append("Stack components (= histogram entries, bottom of the stack first,")
summary_lines.append("i.e. the declared top-down order reversed -- what the figure and the per-bin table use):")
# No job_selected_by_key passed: the ordering ignores it (it is accepted only to
# mirror the legend call), and it is undefined when the stacks were not drawn.
for component in order_components_for_stack(SIGNAL_BACKGROUND_COMPONENTS):
    summary_lines.append(f"  {component['key']:<24s} {component_counts[component['key']]:6d} clusters")
summary_lines.append(f"  {'TOTAL IN STACK':<24s} {sum(component_counts.values()):6d} clusters")
summary_lines.append("")
# Reported only when the pairing actually ran. Without it every cluster has
# pair_completeness=None and the split would read "0 high completeness, all
# contaminated" -- a result, and a wrong one, where the honest output is silence.
if NEEDS_PAIRING:
    n_high = len(SIGNAL_BACKGROUND_COMPONENTS_NUMU_QUALITY[0]['select'](job_true_var_records))
    n_cont = len(SIGNAL_BACKGROUND_COMPONENTS_NUMU_QUALITY[1]['select'](job_true_var_records))
    summary_lines.append("")
    summary_lines.append(f"NumuCCQuality split of the signal band "
                         f"(completeness AND purity > {NUMU_QUALITY_THRESHOLD:.0%}, "
                         f"paired against the {RECO_SELECTION_BEAM} reco clusters):")
    summary_lines.append(f"  high completeness        {n_high:6d} clusters")
    summary_lines.append(f"  contaminated             {n_cont:6d} clusters")
    summary_lines.append(f"  (the two partition the {component_counts['numu_CC_in_volume']} "
                         f"numu CC in-volume clusters exactly)")
else:
    summary_lines.append("")
    summary_lines.append("NumuCCQuality split: NOT computed (no variant asked for it, so the")
    summary_lines.append("  1-to-1 pairing was skipped -- see NEEDS_PAIRING).")
summary_lines.append("")
summary_lines.append(f"True clusters in no stack component: "
                     f"{len(job_true_var_records) - sum(component_counts.values())}")
summary_lines.append("  (cosmic clusters, and neutrinos of a channel or vertex volume that no")
summary_lines.append("   component claims -- they join the stack as components are added to")
summary_lines.append("   SIGNAL_BACKGROUND_COMPONENTS in draw_signal_background.py)")
summary_lines.append("")
summary_lines.append("=" * 80)
summary_lines.append("JOB RUNTIME")
summary_lines.append("=" * 80)
summary_lines.append(f"Job started at:  {datetime.fromtimestamp(job_start_time).strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Job finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')}")
summary_lines.append(f"Total job runtime: {timedelta(seconds=int(job_runtime))} ({job_runtime:.1f} seconds)")
summary_lines.append(f"  of which one-time input staging: {timedelta(seconds=int(staging_seconds))} ({staging_seconds:.1f} s)   -- ~0 on a re-run / the tagger sample")
summary_lines.append("=" * 80)

with open(job_output_dir / "summary.txt", "w") as f:
    f.write("\n".join(summary_lines) + "\n")

print(f"\nJob finished at: {job_finish_dt.strftime('%Y-%m-%d %H:%M:%S')} (runtime: {job_runtime:.1f}s)")
print(f"Job summary written to: {job_output_dir / 'summary.txt'}")
